# 🎬 Comprehensive IMDb Web Scraper & Movie Data Analysis
### *A complete guide to scraping modern dynamic web applications, bypassing anti-bot challenges, and analyzing movie datasets.*

---

## 📌 Project Overview
Scraping modern high-traffic web applications like **IMDb** presents modern data extraction challenges:
1. **Dynamic Rendering & Client-Side Hydration**: IMDb uses Next.js with embedded JSON (`__NEXT_DATA__` and `<script type="application/ld+json">`).
2. **Anti-Bot & WAF Protections**: Web Application Firewalls (AWS WAF) deploy JS challenges on standard non-browser requests.
3. **Structured vs Brittle Selectors**: High-quality scrapers prioritize structured Schema.org JSON-LD and internal state before relying on fragile CSS/XPath selectors.

---

## 🛠️ Capabilities of this Scraper
- ⚡ **Instant Search**: Fuzzy search for movies, shows, and stars via IMDb's Suggestion API (no browser needed, ultra-fast).
- 🏆 **Top 250 Charts**: Full Top 250 Movies and Top 250 TV Shows with ranks, ratings, vote counts, durations, and certificates.
- 🔍 **Deep Title Metadata**: Director, Writers, Cast, Storylines, Keywords, Content Rating, Metascore, Production Budget, and Worldwide Box Office.
- 📊 **Pandas & Data Science Ready**: Conversion to clean DataFrames and export to CSV/JSON.

In [ ]:
# Step 1: Import dependencies and IMDb Scraper module
import os
import sys
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add local package path
sys.path.insert(0, os.path.abspath('.'))
from imdb_scraper import IMDbScraper

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Dependencies loaded successfully!")

## 1. 🔍 Instant Search (IMDb Suggestion API)
No browser rendering is required for search queries. IMDb provides a lightweight endpoint for instant auto-complete suggestions.

In [ ]:
# Initialize scraper instance
scraper = IMDbScraper(headless=True)

# Instant search for any title or person
query = "Oppenheimer"
results = scraper.search(query, limit=5)

# Convert to DataFrame for neat display
df_search = pd.DataFrame(results)
df_search[["id", "title", "year", "type", "stars"]]

## 2. 🏆 Scraping IMDb Top 250 Movies Chart
We scrape the Top Movies chart using our stealth browser engine which captures structured **JSON-LD ItemList** data.

In [ ]:
# Scrape Top 25 Movies (or up to 250)
top_movies = scraper.get_top_250_movies(limit=25)

# Convert to clean DataFrame
df_top = scraper.to_dataframe(top_movies)
print(f"Total movies scraped: {len(df_top)}")
df_top[["rank", "title", "rating", "vote_count", "duration", "genres"]].head(10)

## 3. 📺 Scraping IMDb Top TV Shows
Similarly, we can scrape the Top 250 TV Shows chart.

In [ ]:
# Scrape Top 15 TV Shows
top_tv = scraper.get_top_250_tv_shows(limit=15)
df_tv = scraper.to_dataframe(top_tv)
df_tv[["rank", "title", "rating", "vote_count", "genres"]].head(10)

## 4. 🎬 Extracting Deep Movie Metadata & Box Office Details
Let's inspect the deep metadata for *The Shawshank Redemption* (`tt0111161`) and *Inception* (`tt1375666`).

In [ ]:
# Deep scrape a single movie
movie_id = "tt0111161"  # The Shawshank Redemption
details = scraper.get_movie_details(movie_id)

print("=" * 60)
print(f"Title:        {details.get('title')}")
print(f"Release Date: {details.get('release_date')}")
print(f"Rating:       {details.get('rating')}/10 ({details.get('vote_count')} votes)")
print(f"Metascore:    {details.get('metascore')}")
print(f"Duration:     {details.get('duration')} ({details.get('runtime_minutes')} mins)")
print(f"Certificate:  {details.get('content_rating')}")
print(f"Directors:    {details.get('directors')}")
print(f"Stars:        {details.get('stars')}")
print(f"Budget:       {details.get('budget')}")
print(f"Gross World:  {details.get('gross_worldwide')}")
print(f"Storyline:    {details.get('description')}")
print("=" * 60)

## 5. 📊 Exploratory Data Analysis (EDA) on Scraped Data
Let's visualize the ratings distribution and genre frequencies of the scraped top movies.

In [ ]:
# Visualizing distributions
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# 1. Rating Distribution
sns.histplot(df_top["rating"].dropna(), bins=8, kde=True, ax=axes[0], color="royalblue")
axes[0].set_title("Distribution of IMDb Top Ratings", fontsize=14, fontweight="bold")
axes[0].set_xlabel("IMDb Rating")
axes[0].set_ylabel("Count")

# 2. Top Genres
genre_series = df_top["genres"].dropna().str.split(", ").explode()
top_genres = genre_series.value_counts().head(8)

sns.barplot(x=top_genres.values, y=top_genres.index, ax=axes[1], hue=top_genres.index, palette="viridis", legend=False)
axes[1].set_title("Most Frequent Genres in Top Movies", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Number of Movies")

plt.tight_layout()
plt.show()

## 6. 💾 Exporting Scraped Data to CSV & JSON
Save the dataset to files for machine learning and reporting.

In [ ]:
# Export dataset
scraper.save_to_csv(top_movies, "scraped_top_movies.csv")
scraper.save_to_json(top_movies, "scraped_top_movies.json")

# Close scraper browser context
scraper.close()
print("Scraper closed and files exported successfully!")